In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd

G = ox.load_graphml(
    "turku_bike_split.graphml",
    edge_dtypes={
        "flooded": lambda x: str(x).lower() == "true"
    }
)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"CRS: {G.graph['crs']}")

Nodes: 53761
Edges: 115385
CRS: EPSG:3067


In [2]:
flooded_count = sum(
    data.get("flooded", False)
    for _, _, _, data in G.edges(keys=True, data=True)
)

print(f"Flooded edges: {flooded_count}")

Flooded edges: 36754


In [3]:
G_normal = G
G_flooded = G.copy()

edges_to_remove = [
    (u, v, key)
    for u, v, key, data in G_flooded.edges(
        keys=True, data=True
    )
    if data.get("flooded", False)
]

G_flooded.remove_edges_from(edges_to_remove)

print("Normal edges:", G_normal.number_of_edges())
print("Flooded edges:", G_flooded.number_of_edges())

Normal edges: 115385
Flooded edges: 78631


In [4]:
# Replace with two actual locations in your study area

start_lon = 22.25
start_lat = 60.46

school_lon = 22.358744
school_lat = 60.44546

In [5]:
from pyproj import Transformer

transformer = Transformer.from_crs(
    "EPSG:4326",
    G.graph["crs"],
    always_xy=True
)


start_x, start_y = transformer.transform(
    start_lon, start_lat
)

school_x, school_y = transformer.transform(
    school_lon, school_lat
)

In [6]:
start_node = ox.distance.nearest_nodes(
    G_normal,
    X=start_x,
    Y=start_y
)

school_node = ox.distance.nearest_nodes(
    G_normal,
    X=school_x,
    Y=school_y
)

def node_status(node, G_flooded):
    if node not in G_flooded:
        return "not_in_graph"

    if G_flooded.degree(node) == 0:
        return "in flooded area"

    return "not in flooded area"

print("Start:", node_status(start_node, G_flooded))
print("School:", node_status(school_node, G_flooded))

Start: not in flooded area
School: not in flooded area


In [7]:
# Normal shortest route
try:
    route_normal = nx.shortest_path(
        G_normal,
        source=start_node,
        target=school_node,
        weight="length"
    )
except nx.NetworkXNoPath:
    route_normal = None

# Flood-avoiding shortest route
try:
    route_flooded = nx.shortest_path(
        G_flooded,
        source=start_node,
        target=school_node,
        weight="length"
    )
except nx.NetworkXNoPath:
    route_flooded = None

print("Normal route:", route_normal is not None)
print("Flood-avoiding route:", route_flooded is not None)

Normal route: True
Flood-avoiding route: False


In [8]:
if route_normal is not None:
    normal_gdf = ox.routing.route_to_gdf(
        G_normal, route_normal, weight="length"
    )

if route_flooded is not None:
    flooded_gdf = ox.routing.route_to_gdf(
        G_flooded, route_flooded, weight="length"
    )

if route_normal is not None:
    normal_length = normal_gdf["length"].sum()
    print(f"Normal route: {normal_length:.0f} m")

if route_flooded is not None:
    flooded_length = flooded_gdf["length"].sum()
    print(f"Flood-avoiding route: {flooded_length:.0f} m")

Normal route: 7655 m


In [9]:
import networkx as nx

def network_stats(G, name):
    G_test = G.copy()

    # Remove leftover isolated nodes
    G_test.remove_nodes_from(list(nx.isolates(G_test)))

    components = sorted(
        nx.weakly_connected_components(G_test),
        key=len,
        reverse=True
    )

    print(f"\n{name}")
    print("Nodes:", G_test.number_of_nodes())
    print("Edges:", G_test.number_of_edges())
    print("Weak components:", len(components))
    print("Largest component:", len(components[0]))
    print(
        "Largest component (%):",
        round(
            len(components[0]) /
            G_test.number_of_nodes() * 100, 1
        )
    )

network_stats(G_normal, "Normal network")
network_stats(G_flooded, "Flooded network")


Normal network
Nodes: 53761
Edges: 115385
Weak components: 424
Largest component: 52124
Largest component (%): 97.0

Flooded network
Nodes: 50190
Edges: 78631
Weak components: 11768
Largest component: 4376
Largest component (%): 8.7
